In [1]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:
from getpass import getpass

# Securely read the token
github_token = getpass('GitHub token: ')
github_user = 'iremcesur'
repo_name = 'CENG467_Midterm_310201051'

!git clone https://{github_token}@github.com/{github_user}/{repo_name}.git
%cd {repo_name}

!git config user.email "iremcesur310201051@gmail.com"
!git config user.name "iremcesur"
!git config pull.rebase false

GitHub token: ··········
Cloning into 'CENG467_Midterm_310201051'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 53 (delta 15), reused 34 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 1.52 MiB | 6.33 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/CENG467_Midterm_310201051


In [3]:
import random
import numpy as np
import torch
import os

# Reproducibility — same seed across all questions
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Install summarization-specific libraries
# - rouge_score: ROUGE-1/2/L
# - sacrebleu: BLEU
# - bert_score: contextual semantic similarity
# - networkx + nltk: TextRank (graph-based extractive summarization)
!pip install -q datasets transformers evaluate rouge_score sacrebleu bert_score networkx nltk

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print(f"✓ Seeds set to {SEED}")
print(f"✓ Libraries ready")
print(f"✓ Device: {device}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00
✓ Seeds set to 42
✓ Libraries ready
✓ Device: cuda


In [4]:
from datasets import load_dataset

# Load CNN/DailyMail (v3 is the standard for summarization benchmarks)
# This downloads ~600 MB on first run; cached afterwards.
print("Loading CNN/DailyMail v3.0.0 ...")
cnn_dm = load_dataset("cnn_dailymail", "3.0.0")

print(f"\nDataset structure: {cnn_dm}")
print(f"\nColumns: {cnn_dm['train'].column_names}")
print(f"Train: {len(cnn_dm['train']):,} examples")
print(f"Val:   {len(cnn_dm['validation']):,} examples")
print(f"Test:  {len(cnn_dm['test']):,} examples")

# Look at one example to understand structure
sample = cnn_dm['test'][0]
print(f"\n--- Sample example ---")
print(f"Article (first 400 chars):\n{sample['article'][:400]}...")
print(f"\nReference summary (highlights):\n{sample['highlights']}")
print(f"\nArticle length: {len(sample['article']):,} chars")
print(f"Summary length: {len(sample['highlights']):,} chars")

Loading CNN/DailyMail v3.0.0 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]


Dataset structure: DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

Columns: ['article', 'highlights', 'id']
Train: 287,113 examples
Val:   13,368 examples
Test:  11,490 examples

--- Sample example ---
Article (first 400 chars):
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also acce...

Reference summary (highlights):
Membership gives the ICC jurisdiction over allege

In [5]:
import pandas as pd

# Take a 100-example random subset of the test set.
# Why 100? It's a standard size in academic summarization papers when
# compute is constrained, and large enough for statistically meaningful
# corpus-level metrics (ROUGE, BLEU, METEOR are averaged across examples).
SUBSET_SIZE = 100

# Convert to pandas for easy manipulation, then sample with our fixed seed
test_df = pd.DataFrame(cnn_dm['test'])
test_subset = test_df.sample(n=SUBSET_SIZE, random_state=SEED).reset_index(drop=True)

# Filter out any extreme outliers (very short/long articles bias evaluation)
print(f"Article length distribution (chars):")
print(test_subset['article'].str.len().describe())
print(f"\nSummary length distribution (chars):")
print(test_subset['highlights'].str.len().describe())

# Save for reproducibility — both summarization models will use this exact subset
os.makedirs('data/processed', exist_ok=True)
test_subset.to_csv('data/processed/cnndm_test100.csv', index=False)
print(f"\n✓ Saved 100-example subset to data/processed/cnndm_test100.csv")

Article length distribution (chars):
count     100.000000
mean     3677.010000
std      1601.527122
min      1115.000000
25%      2388.750000
50%      3667.500000
75%      4567.750000
max      7756.000000
Name: article, dtype: float64

Summary length distribution (chars):
count    100.000000
mean     294.690000
std       97.981826
min      108.000000
25%      215.750000
50%      280.500000
75%      338.250000
max      591.000000
Name: highlights, dtype: float64

✓ Saved 100-example subset to data/processed/cnndm_test100.csv


In [6]:
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

def textrank_summarize(text, num_sentences=3):
    """
    TextRank extractive summarization.

    Steps:
        1. Split article into sentences.
        2. Build TF-IDF vectors for each sentence.
        3. Compute pairwise cosine similarity → adjacency matrix.
        4. Run PageRank on this graph; nodes = sentences, weights = similarities.
        5. Pick the top-`num_sentences` sentences by PageRank score.
        6. Return them in their original document order (preserves narrative flow).

    Args:
        text: full article string.
        num_sentences: how many sentences to keep (3 matches typical summary length).
    """
    sentences = sent_tokenize(text)

    # Edge case: very short articles
    if len(sentences) <= num_sentences:
        return ' '.join(sentences)

    # TF-IDF on the sentences themselves (mini in-document corpus)
    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        sentence_vectors = vectorizer.fit_transform(sentences)
    except ValueError:
        # Empty vocabulary (all stopwords) — fall back to first N sentences
        return ' '.join(sentences[:num_sentences])

    # Pairwise cosine similarity → fully-connected weighted graph
    sim_matrix = cosine_similarity(sentence_vectors)
    np.fill_diagonal(sim_matrix, 0)  # zero out self-loops

    # PageRank over the similarity graph
    nx_graph = nx.from_numpy_array(sim_matrix)
    try:
        scores = nx.pagerank(nx_graph, max_iter=500)
    except nx.PowerIterationFailedConvergence:
        # Fallback: rank by sum of similarities to other sentences
        scores = {i: float(sim_matrix[i].sum()) for i in range(len(sentences))}

    # Pick top-K sentences by score, but return them in their *original* order
    ranked_indices = sorted(scores, key=scores.get, reverse=True)[:num_sentences]
    selected_indices = sorted(ranked_indices)

    return ' '.join(sentences[i] for i in selected_indices)


# Quick sanity check on one example
sample_article = test_subset['article'].iloc[0]
sample_summary_tr = textrank_summarize(sample_article, num_sentences=3)
print(f"Article (first 300 chars): {sample_article[:300]}...\n")
print(f"TextRank summary:\n{sample_summary_tr}")
print(f"\nReference summary:\n{test_subset['highlights'].iloc[0]}")

Article (first 300 chars): Down Augusta way they say the azaleas are in full bloom, which is more than can be said for England’s Justin Rose. A bruising Florida swing last month saw the Englishman fall outside the world’s top 10. For a player who has been virtually a fixture in the top five for the last three years it was cer...

TextRank summary:
‘Over the past two weeks I feel like I’ve done some good work and whether I finish well or not here I feel like I’m going in the right direction again. So we’ve corrected the faults and I’ve gone back to the old putter.’ Phil Mickelson enjoyed his best round in months with a 66 on Thursday . Paul Casey, like Mickelson another former winner of this event, celebrated his last-gasp Masters invitation with a fine round of 68 notable for two eagle threes.

Reference summary:
Justin Rose bounced back from Florida misery by carding 69 in Houston .
Three-time Masters champion Phil Mickelson enjoyed return to form .
Paul Casey celebrated last-gasp Mas

In [7]:
import time
from tqdm.notebook import tqdm

print(f"Generating TextRank summaries for {SUBSET_SIZE} articles...")
start = time.time()

textrank_summaries = []
for article in tqdm(test_subset['article']):
    summary = textrank_summarize(article, num_sentences=3)
    textrank_summaries.append(summary)

elapsed = time.time() - start
print(f"\n✓ Done in {elapsed:.1f}s ({elapsed/SUBSET_SIZE:.2f}s per article)")
print(f"\nMean summary length: {np.mean([len(s) for s in textrank_summaries]):.0f} chars")
print(f"Mean reference length: {np.mean([len(s) for s in test_subset['highlights']]):.0f} chars")

Generating TextRank summaries for 100 articles...


  0%|          | 0/100 [00:00<?, ?it/s]


✓ Done in 1.3s (0.01s per article)

Mean summary length: 452 chars
Mean reference length: 295 chars


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# DistilBART-cnn-12-6: a distilled BART specifically fine-tuned on
# CNN/DailyMail. ~306M parameters (vs ~406M for full BART), ~50% faster
# inference, with similar ROUGE scores. Fits comfortably in T4 memory.
BART_MODEL = "sshleifer/distilbart-cnn-12-6"

print(f"Loading {BART_MODEL} (this can take 30-60s on first run)...")

tokenizer_bart = AutoTokenizer.from_pretrained(BART_MODEL)
model_bart = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL).to(device)
model_bart.eval()  # inference only — no training, no gradients

# Print parameter count for the report
n_params = sum(p.numel() for p in model_bart.parameters())
print(f"\n✓ Model ready")
print(f"  Parameters: {n_params:,}")
print(f"  Encoder layers: {model_bart.config.encoder_layers}")
print(f"  Decoder layers: {model_bart.config.decoder_layers}")


def bart_summarize(article_text, max_length=130, min_length=30, num_beams=4):
    """
    Summarize a single article using DistilBART.
    Uses the model directly (no pipeline) to avoid HF version incompatibilities.
    Beam search (num_beams=4) is the standard decoding for CNN/DM evaluation.
    """
    inputs = tokenizer_bart(
        article_text,
        max_length=1024,        # BART's max input length
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        summary_ids = model_bart.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            min_length=min_length,
            num_beams=num_beams,
            length_penalty=2.0,           # encourage slightly longer summaries
            no_repeat_ngram_size=3,       # avoid trigram repetition
            early_stopping=True,
        )

    return tokenizer_bart.decode(summary_ids[0], skip_special_tokens=True)


# Sanity check on the same article TextRank just summarized
test_article = test_subset['article'].iloc[0]
sample_bart = bart_summarize(test_article)
print(f"\nDistilBART summary of test article:\n{sample_bart}")

Loading sshleifer/distilbart-cnn-12-6 (this can take 30-60s on first run)...


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



✓ Model ready
  Parameters: 408,451,072
  Encoder layers: 12
  Decoder layers: 6

DistilBART summary of test article:
 Justin Rose hit 17 out of 18 greens in regulation and signed for a 69 at the Shell Houston Open . The Englishman fell outside the world’s top 10 last month after a bruising Florida swing . Phil Mickelson enjoyed his best round in months with a 66 . Paul Casey celebrated his last-gasp Masters invitation with two eaglerees of 68 .


In [10]:
print(f"Generating DistilBART summaries for {SUBSET_SIZE} articles...")
print("(Beam search with 4 beams; expect ~3-5 minutes total)")
start = time.time()

bart_summaries = []
for article in tqdm(test_subset['article']):
    summary = bart_summarize(article)
    bart_summaries.append(summary)

elapsed = time.time() - start
print(f"\n✓ Done in {elapsed:.1f}s ({elapsed/SUBSET_SIZE:.2f}s per article)")
print(f"\nMean summary length: {np.mean([len(s) for s in bart_summaries]):.0f} chars")
print(f"Mean reference length: {np.mean([len(s) for s in test_subset['highlights']]):.0f} chars")

# Show same article side by side with all three
idx = 0
print(f"\n{'='*70}\nSIDE-BY-SIDE COMPARISON (article #{idx})\n{'='*70}")
print(f"\n--- Reference ---\n{test_subset['highlights'].iloc[idx]}")
print(f"\n--- TextRank (extractive) ---\n{textrank_summaries[idx]}")
print(f"\n--- DistilBART (abstractive) ---\n{bart_summaries[idx]}")

Generating DistilBART summaries for 100 articles...
(Beam search with 4 beams; expect ~3-5 minutes total)


  0%|          | 0/100 [00:00<?, ?it/s]


✓ Done in 110.5s (1.11s per article)

Mean summary length: 380 chars
Mean reference length: 295 chars

SIDE-BY-SIDE COMPARISON (article #0)

--- Reference ---
Justin Rose bounced back from Florida misery by carding 69 in Houston .
Three-time Masters champion Phil Mickelson enjoyed return to form .
Paul Casey celebrated last-gasp Masters invitation with fine round of 68 .

--- TextRank (extractive) ---
‘Over the past two weeks I feel like I’ve done some good work and whether I finish well or not here I feel like I’m going in the right direction again. So we’ve corrected the faults and I’ve gone back to the old putter.’ Phil Mickelson enjoyed his best round in months with a 66 on Thursday . Paul Casey, like Mickelson another former winner of this event, celebrated his last-gasp Masters invitation with a fine round of 68 notable for two eagle threes.

--- DistilBART (abstractive) ---
 Justin Rose hit 17 out of 18 greens in regulation and signed for a 69 at the Shell Houston Open . The En

In [11]:
import evaluate

# Load all evaluation metrics. First run downloads them; cached afterwards.
print("Loading evaluation metrics (first run downloads from HuggingFace)...")
rouge       = evaluate.load("rouge")
bleu        = evaluate.load("sacrebleu")
meteor      = evaluate.load("meteor")
bertscore   = evaluate.load("bertscore")
print("✓ Metrics ready")


def evaluate_summaries(predictions, references, model_name):
    """
    Compute ROUGE-1/2/L, BLEU, METEOR, BERTScore for a list of (prediction, reference)
    pairs. Returns a dict for easy aggregation.
    """
    print(f"\nEvaluating {model_name}...")

    # ROUGE — n-gram overlap; the standard summarization metric.
    # use_stemmer=True is the convention for CNN/DM evaluation.
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,
    )

    # BLEU — sacrebleu expects references as list-of-lists (multiple refs allowed).
    bleu_score = bleu.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )

    # METEOR — alignment-based, handles synonyms and stemming.
    meteor_score = meteor.compute(
        predictions=predictions,
        references=references,
    )

    # BERTScore — uses contextual embeddings; closer to semantic similarity.
    # We average the F1 across all examples.
    bert_scores = bertscore.compute(
        predictions=predictions,
        references=references,
        lang='en',
        verbose=False,
        rescale_with_baseline=True,   # makes scores more interpretable
    )
    bert_f1_mean = float(np.mean(bert_scores['f1']))

    results = {
        'model': model_name,
        'rouge1':    float(rouge_scores['rouge1']),
        'rouge2':    float(rouge_scores['rouge2']),
        'rougeL':    float(rouge_scores['rougeL']),
        'bleu':      float(bleu_score['score']) / 100.0,   # sacrebleu returns 0-100
        'meteor':    float(meteor_score['meteor']),
        'bertscore_f1': bert_f1_mean,
    }

    return results


# Compute metrics for both systems
references = test_subset['highlights'].tolist()
results_textrank = evaluate_summaries(textrank_summaries, references, "TextRank")
results_bart     = evaluate_summaries(bart_summaries,    references, "DistilBART")

# Side-by-side display
import pandas as pd
df_results = pd.DataFrame([results_textrank, results_bart]).set_index('model')
df_results = df_results.round(4)
print("\n" + "=" * 70)
print("SUMMARIZATION METRICS COMPARISON")
print("=" * 70)
print(df_results.to_string())

# Quick interpretation
print(f"\n--- Improvements (DistilBART relative to TextRank) ---")
for col in df_results.columns:
    diff = df_results.loc['DistilBART', col] - df_results.loc['TextRank', col]
    pct  = diff / df_results.loc['TextRank', col] * 100
    sign = "+" if diff >= 0 else ""
    print(f"  {col:<15} {sign}{diff:.4f}  ({sign}{pct:.1f}%)")

Loading evaluation metrics (first run downloads from HuggingFace)...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✓ Metrics ready

Evaluating TextRank...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Evaluating DistilBART...

SUMMARIZATION METRICS COMPARISON
            rouge1  rouge2  rougeL    bleu  meteor  bertscore_f1
model                                                           
TextRank    0.3557  0.1463  0.2328  0.0954  0.3338        0.2158
DistilBART  0.4332  0.2129  0.3068  0.1695  0.4167        0.3308

--- Improvements (DistilBART relative to TextRank) ---
  rouge1          +0.0775  (+21.8%)
  rouge2          +0.0666  (+45.5%)
  rougeL          +0.0740  (+31.8%)
  bleu            +0.0741  (+77.7%)
  meteor          +0.0829  (+24.8%)
  bertscore_f1    +0.1150  (+53.3%)


In [12]:
import re

# We need 3 qualitatively distinct examples covering the trade-offs:
#   (a) BART substantially better than TextRank — abstraction wins
#   (b) BART comparable or worse than TextRank — extractive can be enough
#   (c) Both struggle — long/complex articles where neither captures the gist

# Compute a per-example ROUGE-L score for each system to rank them
def per_example_rouge(predictions, references):
    scores = []
    for pred, ref in zip(predictions, references):
        r = rouge.compute(predictions=[pred], references=[ref], use_stemmer=True)
        scores.append(r['rougeL'])
    return np.array(scores)

print("Computing per-example ROUGE-L scores...")
tr_scores   = per_example_rouge(textrank_summaries, references)
bart_scores = per_example_rouge(bart_summaries,    references)

# Difference (positive => BART better)
diff = bart_scores - tr_scores

# Pick three examples with distinct qualitative profiles
idx_bart_wins        = int(np.argmax(diff))                    # BART much better
idx_textrank_wins    = int(np.argmin(diff))                    # TextRank better (rare)
idx_both_struggle    = int(np.argmin(np.maximum(tr_scores, bart_scores)))  # both low

selected_indices = [idx_bart_wins, idx_textrank_wins, idx_both_struggle]
labels = [
    "BART > TextRank (abstraction wins)",
    "TextRank ≥ BART (extractive sufficient)",
    "Both struggle (genuinely hard article)",
]

print("\n" + "=" * 80)
print("THREE QUALITATIVE EXAMPLES FOR THE REPORT")
print("=" * 80)

qualitative_examples = []
for label, idx in zip(labels, selected_indices):
    print(f"\n{'='*80}")
    print(f"[{label}]  (index {idx})")
    print(f"  TextRank ROUGE-L: {tr_scores[idx]:.3f}")
    print(f"  DistilBART ROUGE-L: {bart_scores[idx]:.3f}")
    print(f"{'='*80}")

    article = test_subset['article'].iloc[idx]
    reference = test_subset['highlights'].iloc[idx]
    tr_sum    = textrank_summaries[idx]
    bart_sum  = bart_summaries[idx]

    print(f"\n--- Article (first 400 chars) ---\n{article[:400]}...")
    print(f"\n--- Reference summary ---\n{reference}")
    print(f"\n--- TextRank summary ---\n{tr_sum}")
    print(f"\n--- DistilBART summary ---\n{bart_sum}")

    qualitative_examples.append({
        "category": label,
        "index": idx,
        "article_excerpt": article[:600],
        "reference": reference,
        "textrank": tr_sum,
        "bart": bart_sum,
        "textrank_rougeL": float(tr_scores[idx]),
        "bart_rougeL":     float(bart_scores[idx]),
    })

print("\n✓ Selected 3 qualitative examples for the report")

Computing per-example ROUGE-L scores...

THREE QUALITATIVE EXAMPLES FOR THE REPORT

[BART > TextRank (abstraction wins)]  (index 8)
  TextRank ROUGE-L: 0.211
  DistilBART ROUGE-L: 0.629

--- Article (first 400 chars) ---
Apple offers a range of ways to unlock its devices from PINs to passwords and fingerprints. But you could soon use a selfie to gain access to your apps and messages thanks to Apple's latest patent. The filing details a system of scanning a user's face with the front-facing camera when the handset is moved into a certain position, and automatically unlocking the device if the image matches one on f...

--- Reference summary ---
The patent was filed in March 2011 and awarded to Apple earlier this week .
It details a method of scanning a user's face using the front-facing camera .
If the scanned face matches with a photo that was previously taken and stored, the phone unlocks automatically .
Android Lollipop already has a similar feature called 'Trusted face'

--- TextRan

In [13]:
import json
import os

os.makedirs('q3_summarization/results', exist_ok=True)

# Aggregate all Q3 outputs into one structured JSON
results = {
    'task': 'summarization',
    'dataset': 'cnn_dailymail v3.0.0',
    'subset_size': SUBSET_SIZE,
    'random_seed': SEED,
    'models': {
        'textrank': {
            'description': '3-sentence extractive via TF-IDF + PageRank',
            'mean_summary_length_chars': float(np.mean([len(s) for s in textrank_summaries])),
            **{k: v for k, v in results_textrank.items() if k != 'model'},
        },
        'distilbart': {
            'description': 'sshleifer/distilbart-cnn-12-6, beam search (4 beams)',
            'parameters': int(n_params),
            'decoding': {
                'num_beams': 4,
                'max_length': 130,
                'min_length': 30,
                'length_penalty': 2.0,
                'no_repeat_ngram_size': 3,
            },
            'mean_summary_length_chars': float(np.mean([len(s) for s in bart_summaries])),
            **{k: v for k, v in results_bart.items() if k != 'model'},
        },
    },
    'mean_reference_length_chars': float(np.mean([len(s) for s in references])),
    'qualitative_examples': qualitative_examples,
}

with open('q3_summarization/results/q3_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save the raw summaries too (useful if we want to re-evaluate later)
out_df = pd.DataFrame({
    'article':   test_subset['article'].values,
    'reference': test_subset['highlights'].values,
    'textrank':  textrank_summaries,
    'distilbart': bart_summaries,
    'rouge_l_textrank':   tr_scores,
    'rouge_l_distilbart': bart_scores,
})
out_df.to_csv('q3_summarization/results/q3_predictions.csv', index=False)

print("✓ Saved q3_results.json and q3_predictions.csv")
!ls -la q3_summarization/results/

✓ Saved q3_results.json and q3_predictions.csv
total 492
drwxr-xr-x 2 root root   4096 May  6 08:07 .
drwxr-xr-x 3 root root   4096 May  6 08:07 ..
-rw-r--r-- 1 root root 486942 May  6 08:07 q3_predictions.csv
-rw-r--r-- 1 root root   7439 May  6 08:07 q3_results.json


In [14]:
# Git commit & push
!git pull origin main --no-edit
!git add q3_summarization/
!git commit -m "Q3: Summarization — TextRank vs DistilBART (ROUGE-L 0.23 vs 0.31, +31.8%)"
!git push origin main

From https://github.com/iremcesur/CENG467_Midterm_310201051
 * branch            main       -> FETCH_HEAD
Already up to date.
[main d34659c] Q3: Summarization — TextRank vs DistilBART (ROUGE-L 0.23 vs 0.31, +31.8%)
 2 files changed, 442 insertions(+)
 create mode 100644 q3_summarization/results/q3_predictions.csv
 create mode 100644 q3_summarization/results/q3_results.json
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 157.12 KiB | 3.74 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/iremcesur/CENG467_Midterm_310201051.git
   2bf4db8..d34659c  main -> main
